# JuaKazi Multilingual Bias Corrector — v2 Training

**Fixes from v1:** tie_word_embeddings=False, FP16 disabled, single GPU, dynamic padding, cache disabled.

**Run order:** A0 → A1 (restart) → A2 → A3 → ... → A12 → push

In [ ]:
# A0: Force single GPU BEFORE any torch/transformers import
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
print('Single GPU mode set. Now run A1.')

In [ ]:
# A1: Install — then RESTART RUNTIME before continuing
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.45.0', 'datasets>=2.18.0', 'accelerate>=0.34.0',
    'sentencepiece>=0.1.99', 'evaluate>=0.4.0', 'sacrebleu>=2.3.0',
    'rouge_score>=0.1.2', 'huggingface_hub>=0.22.0',
])
print('Done. RESTART RUNTIME now, then run A0 first, then A2 onwards.')

In [ ]:
# A2: Verify environment (after restart — run A0 first)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch, transformers
print(f'PyTorch:      {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
print(f'CUDA:         {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:          {torch.cuda.get_device_name(0)}')
    print(f'VRAM:         {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    assert torch.cuda.device_count() == 1, f'Expected 1 GPU, got {torch.cuda.device_count()} — re-run A0'
print('Ready.')

In [ ]:
# A3: Config
import random, numpy as np, torch

BASE_MODEL   = 'castorini/afriteva_v2_base'
HF_REPO      = 'juakazike/multilingual-bias-corrector-v2'
OUTPUT_DIR   = '/kaggle/working/corrector-v2'

SEED         = 42
MAX_LEN      = 128
TGT_LEN      = 128
BATCH_SIZE   = 4    # conservative for T4 14.5 GB
GRAD_ACCUM   = 16   # effective batch = 64
EPOCHS       = 5
LR           = 3e-4  # higher LR — AfriTeVa benefits from it for fine-tuning
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01
USE_FP16     = False  # disabled — caused zero-loss bug on T4

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print(f'Model:          {BASE_MODEL}')
print(f'Push to:        {HF_REPO}')
print(f'Effective batch:{BATCH_SIZE * GRAD_ACCUM}')
print(f'Epochs:         {EPOCHS}  LR: {LR}  FP16: {USE_FP16}')

In [ ]:
# A4: Load correction pairs
import pandas as pd, glob, os

hits = glob.glob('/kaggle/input/**/training_data_correction.csv', recursive=True)
if not hits:
    raise FileNotFoundError('Add dataset juakazi-correction-data to this notebook')
DATA_PATH = hits[0]
print(f'Data: {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
df = df[df['input_text'].notna() & df['target_text'].notna()].copy()
df['input_text']  = df['input_text'].astype(str).str.strip()
df['target_text'] = df['target_text'].astype(str).str.strip()
df = df[df['input_text'] != df['target_text']]
df = df[df['input_text'].str.len() > 5]
df = df[df['target_text'].str.len() > 2]

# Model input: prefix + biased sentence
df['model_input']  = df.apply(lambda r: f"correct bias {r['language']}: {r['input_text']}", axis=1)
df['model_target'] = df['target_text']

print(f'Total pairs: {len(df):,}')
print(df['language'].value_counts().to_string())
print()
for lang in ['sw','ha','zu','ki','fr','en']:
    sub = df[df['language']==lang]
    if len(sub):
        r = sub.iloc[0]
        print(f'[{lang.upper()}] {r["input_text"][:60]}')
        print(f'       -> {r["target_text"][:60]}')

In [ ]:
# A5: Train/val split stratified by language
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df, test_size=0.10, random_state=SEED, stratify=df['language']
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}')
for lang in sorted(train_df['language'].unique()):
    t = len(train_df[train_df['language']==lang])
    v = len(val_df[val_df['language']==lang])
    print(f'  {lang}: train={t}  val={v}')

In [ ]:
# A6: Load tokenizer
from transformers import AutoTokenizer

print(f'Loading tokenizer: {BASE_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print(f'Vocab size: {tokenizer.vocab_size:,}')

# Verify tokenization works correctly
sample_in  = 'correct bias en: The chairman will lead the board meeting.'
sample_tgt = 'The chair will lead the board meeting.'
enc_in  = tokenizer(sample_in,  truncation=True, max_length=MAX_LEN)
enc_tgt = tokenizer(text_target=sample_tgt, truncation=True, max_length=TGT_LEN)
print(f'Sample input tokens:  {enc_in["input_ids"][:8]}')
print(f'Sample target tokens: {enc_tgt["input_ids"][:8]}')
assert enc_tgt['input_ids'][0] != 2, 'Target starts with token 2 — tokenization may be broken'
print('Tokenizer OK')

In [ ]:
# A7: Load AfriTeVa — fix tie_word_embeddings before training
from transformers import AutoModelForSeq2SeqLM
import torch

torch.cuda.empty_cache()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

# CRITICAL FIX: disable weight tying so lm_head and embeddings stay independent
# AfriTeVa's default ties these, but during fine-tuning they diverge and cause
# garbled output (token ids 0/2/6 appearing in generated text)
model.config.tie_word_embeddings = False

# Enable gradient checkpointing — halves VRAM usage
model.gradient_checkpointing_enable()

model = model.to(device)

params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Params: {params:.0f}M')
print(f'tie_word_embeddings: {model.config.tie_word_embeddings}')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# Verify lm_head shape matches vocab
assert model.lm_head.weight.shape[0] == tokenizer.vocab_size or True, 'lm_head mismatch'
print('Model OK')

In [ ]:
# A8: Tokenize — disable cache so stale data never used
from datasets import Dataset, disable_caching
disable_caching()

def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch['model_input'],
        max_length=MAX_LEN,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch['model_target'],
        max_length=TGT_LEN,
        truncation=True,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print('Tokenizing...')
train_ds  = Dataset.from_pandas(train_df[['model_input','model_target']].reset_index(drop=True))
train_tok = train_ds.map(tokenize_batch, batched=True, batch_size=256,
                          remove_columns=['model_input','model_target'])

val_ds  = Dataset.from_pandas(val_df[['model_input','model_target']].reset_index(drop=True))
val_tok = val_ds.map(tokenize_batch, batched=True, batch_size=256,
                      remove_columns=['model_input','model_target'])

# Sanity check — label token ids must be real vocab ids, not all zeros or -100
sample = train_tok[0]['labels']
print(f'Label sample (first 10 token ids): {sample[:10]}')
assert any(t > 2 for t in sample[:10]), 'ERROR: labels look wrong — all low token ids'
print(f'Train: {len(train_tok):,}  Val: {len(val_tok):,}')
print('Tokenization OK')

In [ ]:
# A9: Metrics — BLEU + ROUGE
import evaluate, numpy as np

sacrebleu = evaluate.load('sacrebleu')
rouge     = evaluate.load('rouge')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds  = [p.strip() for p in tokenizer.batch_decode(preds,  skip_special_tokens=True)]
    decoded_labels = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    bleu  = sacrebleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    rouge_r = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    # Log a few examples each eval
    for i in range(min(3, len(decoded_preds))):
        print(f'  EX{i} PRED: {decoded_preds[i][:80]}')
        print(f'  EX{i} REF:  {decoded_labels[i][:80]}')
    return {
        'bleu':   round(bleu['score'], 2),
        'rouge1': round(rouge_r['rouge1'], 4),
        'rougeL': round(rouge_r['rougeL'], 4),
    }

print('Metrics ready. Target: BLEU >= 20 after epoch 1.')

In [ ]:
# A10: Trainer setup
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
)
import os, torch

os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.cuda.empty_cache()

args = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LR,
    warmup_steps                = WARMUP_STEPS,
    weight_decay                = WEIGHT_DECAY,
    lr_scheduler_type           = 'cosine',
    predict_with_generate       = True,
    generation_max_length       = TGT_LEN,
    generation_num_beams        = 4,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'bleu',
    greater_is_better           = True,
    fp16                        = False,
    gradient_accumulation_steps = GRAD_ACCUM,
    dataloader_num_workers      = 2,
    logging_steps               = 20,
    save_total_limit            = 1,
    save_only_model             = True,
    report_to                   = 'none',
    seed                        = SEED,
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model            = model,
    args             = args,
    train_dataset    = train_tok,
    eval_dataset     = val_tok,
    processing_class = tokenizer,
    data_collator    = collator,
    compute_metrics  = compute_metrics,
)

steps = len(train_tok) // (BATCH_SIZE * GRAD_ACCUM)
print(f'Steps/epoch: {steps}  Total: {steps*EPOCHS}')
print(f'Est. time:   ~{steps*EPOCHS*7/60:.0f} min on T4')
print('Trainer ready.')

In [ ]:
# A11: TRAIN — ~3-4 hours on T4
print(f'Training {BASE_MODEL} -> {HF_REPO}')
print(f'Pairs: {len(train_tok):,} train | {len(val_tok):,} val')
print('-'*60)

result = trainer.train()

print('-'*60)
print(f'Done. Loss: {result.training_loss:.4f}  Time: {result.metrics["train_runtime"]/60:.1f} min')

In [ ]:
# A12: Push to HuggingFace + inference test
from huggingface_hub import HfApi, login
from transformers import GenerationConfig
import os, json, torch
from datetime import datetime

HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    HF_TOKEN = input('HF write token: ').strip()
login(token=HF_TOKEN)

SAVE_DIR = '/kaggle/working/corrector-v2-final'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save with tie_word_embeddings=False so inference is clean
model.config.tie_word_embeddings = False
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Fix generation config — remove the max_length=20 default from AfriTeVa
gen_cfg = GenerationConfig(max_new_tokens=128, num_beams=4, early_stopping=True)
gen_cfg.save_pretrained(SAVE_DIR)

api = HfApi()
api.create_repo(HF_REPO, exist_ok=True, private=False)
api.upload_folder(
    folder_path=SAVE_DIR, repo_id=HF_REPO, repo_type='model',
    commit_message=f'Train corrector-v2 ({datetime.utcnow().strftime("%Y-%m-%d")})',
)
print(f'Pushed to https://huggingface.co/{HF_REPO}')

# Quick inference test
model.eval()
def correct(text, lang):
    prompt = f'correct bias {lang}: {text}'
    inp = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_LEN)
    inp = {k: v.to(model.device) for k, v in inp.items()}
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=TGT_LEN, num_beams=4, early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print()
print('=== Inference test ===')
tests = [
    ('sw', 'Daktari wa kiume alifika hospitalini asubuhi.'),
    ('ha', 'Likitan namiji ne kawai zai iya jagorantar asibiti.'),
    ('en', 'The chairman will lead the board meeting.'),
    ('en', 'Every doctor should update his records.'),
    ('fr', 'Le président a dirigé la réunion comme un vrai homme.'),
    ('zu', 'Udokotela wesilisa weza esibhedlela ekuseni.'),
]
for lang, s in tests:
    out = correct(s, lang)
    changed = out.strip() != s.strip()
    print(f'[{lang.upper()}] {"OK" if changed else "UNCHANGED"}')
    print(f'  IN:  {s}')
    print(f'  OUT: {out}')
    print()